# LangChain OpenAI Tools Output Parsers Reference

Developer-facing statements defined in `langchain_core.output_parsers.openai_tools`.

# `parse_tool_call`

Parses one raw OpenAI-style tool call.

```python
parse_tool_call(
    raw_tool_call: dict[str, Any], # Raw tool-call dictionary to parse
    *,
    partial: bool = False, # Whether to accept incomplete JSON arguments
    strict: bool = False, # Value forwarded to JSON decoding strictness
    return_id: bool = True, # Whether to include the tool-call ID
) -> dict[str, Any] | None # Parsed tool call or None
```

Returns `None` when `raw_tool_call` has no `"function"` key.

When `partial=True`, the function parses the raw `"arguments"` value with `parse_partial_json()`. A `JSONDecodeError` or `TypeError` returns `None`.

When `partial=False`, empty or falsey arguments become `{}`. Otherwise, the arguments are decoded with `json.loads(..., strict=strict)`. Invalid JSON raises `OutputParserException` containing the function name, raw arguments, and decoding error.

The parsed name uses an empty string when the raw function name is falsey. Parsed arguments are also replaced with `{}` when the decoded value is falsey.

When `return_id=True`, the raw call's optional `"id"` value is included. The assembled values are passed to the tool-call factory before being returned.

---

# `make_invalid_tool_call`

Creates an invalid-tool-call record from a raw tool call.

```python
make_invalid_tool_call(
    raw_tool_call: dict[str, Any], # Raw tool-call dictionary
    error_msg: str | None, # Error associated with the tool call
) -> InvalidToolCall # Invalid tool-call record
```

The function copies the raw function `"name"` and `"arguments"`, the optional top-level `"id"`, and `error_msg` into `invalid_tool_call()`.

---

# `parse_tool_calls`

Parses multiple raw OpenAI-style tool calls.

```python
parse_tool_calls(
    raw_tool_calls: list[dict[str, Any]], # Raw tool calls to parse
    *,
    partial: bool = False, # Whether to accept incomplete JSON arguments
    strict: bool = False, # Value forwarded to JSON decoding strictness
    return_id: bool = True, # Whether to include tool-call IDs
) -> list[dict[str, Any]] # Parsed tool calls
```

Each item is passed to `parse_tool_call()` with the supplied options. `None` results are omitted.

`OutputParserException` values raised by individual calls are collected. After all calls have been processed, any collected messages are joined with two newline characters and raised as one `OutputParserException`.

# `JsonOutputToolsParser: BaseCumulativeTransformOutputParser[Any]`
Parses OpenAI-style tool calls from chat generations.

## Fields

```python
strict: bool = False # Value forwarded to JSON decoding strictness
return_id: bool = False # Whether returned tool calls retain their IDs
first_tool_only: bool = False # Whether to return only the first tool call
```

When `first_tool_only=False`, parsing normally returns a list. When it is `True`, the first parsed call is returned, or `None` when the parsed list is empty.

## Constructor

```python
JsonOutputToolsParser(
    *,
    strict: bool = False, # Value forwarded to JSON decoding strictness
    return_id: bool = False, # Whether returned tool calls retain their IDs
    first_tool_only: bool = False, # Whether to return only the first tool call
) -> None
```

## Methods

### `parse_result`

Parses tool calls from the first generation.

```python
parse_result(
    self,
    result: list[Generation], # Candidate generations for one model input
    *,
    partial: bool = False, # Whether to accept incomplete JSON arguments
) -> Any # Parsed tool call, list of calls, None, or an empty list
```

The first result must be a `ChatGeneration`; otherwise, the method raises `OutputParserException`.

When the generation contains an `AIMessage` with a truthy `message.tool_calls` value, each existing tool call is copied. When `return_id=False`, the `"id"` entry is removed from each copy.

Otherwise, the method deep-copies `message.additional_kwargs["tool_calls"]` for backward compatibility. If that key is absent, it returns an empty list. Raw calls are passed to `parse_tool_calls()` with `partial`, `strict`, and `return_id`. In this legacy path, each parsed call's `"name"` key is renamed to `"type"`.

When `first_tool_only=True`, the method returns the first parsed call or `None`. Otherwise, it returns the complete parsed list.

### `parse`

Unsupported text-only parsing hook.

```python
parse(
    self,
    text: str, # Text that is not used by this chat-generation parser
) -> Any
```

The implementation always raises `NotImplementedError`.

In [ ]:
from langchain_core.exceptions import OutputParserException # Import the real LangChain parser exception
from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_tools import JsonOutputToolsParser # Import the real tool-call parser
from langchain_core.outputs import ChatGeneration, Generation # Import chat and text generation classes


modern_message = AIMessage( # Create a message using the modern tool_calls field
    content="", # Leave normal message content empty
    tool_calls=[ # Add already-parsed LangChain tool calls
        { # Define the first tool call
            "name": "get_weather", # Provide the tool name
            "args": {"city": "Delhi", "unit": "celsius"}, # Provide parsed tool arguments
            "id": "call_weather_1", # Provide the tool-call ID
            "type": "tool_call", # Identify the dictionary as a tool call
        },
        { # Define the second tool call
            "name": "calculate", # Provide the tool name
            "args": {"first": 10, "second": 5}, # Provide parsed tool arguments
            "id": "call_calculate_1", # Provide the tool-call ID
            "type": "tool_call", # Identify the dictionary as a tool call
        },
    ],
) # Finish creating the modern message

modern_generation = ChatGeneration(message=modern_message) # Wrap the message in a chat generation

parser = JsonOutputToolsParser( # Create the default parser
    strict=False, # Use normal JSON decoding rules
    return_id=False, # Remove IDs from returned tool calls
    first_tool_only=False, # Return every parsed tool call
) # Finish creating the parser

modern_result = parser.parse_result([modern_generation]) # Parse all modern tool calls
print("Modern tool calls:", modern_result) # Display the parsed tool calls

id_parser = JsonOutputToolsParser( # Create a parser that keeps IDs
    return_id=True, # Retain tool-call IDs
    first_tool_only=False, # Return every parsed tool call
) # Finish creating the parser

result_with_ids = id_parser.invoke(modern_message) # Parse through the runnable interface
print("\nModern calls with IDs:", result_with_ids) # Display tool calls with IDs

first_parser = JsonOutputToolsParser( # Create a parser that returns one tool call
    return_id=True, # Retain the selected tool-call ID
    first_tool_only=True, # Return only the first parsed call
) # Finish creating the parser

first_result = first_parser.parse_result([modern_generation]) # Parse only the first tool call
print("\nFirst tool call:", first_result) # Display the first parsed call

async_result = await parser.ainvoke(modern_message) # Parse asynchronously in Jupyter
print("\nAsync result:", async_result) # Display the asynchronous result

legacy_message = AIMessage( # Create a message using raw OpenAI-style tool calls
    content="", # Leave normal message content empty
    additional_kwargs={ # Add provider-specific response data
        "tool_calls": [ # Add raw tool-call dictionaries
            { # Define one raw tool call
                "id": "call_search_1", # Provide the tool-call ID
                "type": "function", # Identify the OpenAI tool-call type
                "function": { # Define the called function
                    "name": "search_products", # Provide the function name
                    "arguments": '{"category": "laptop", "limit": 3}', # Provide JSON arguments
                },
            },
        ],
    },
) # Finish creating the legacy message

legacy_generation = ChatGeneration(message=legacy_message) # Wrap the legacy message

legacy_result = id_parser.parse_result([legacy_generation]) # Parse the legacy raw tool call
print("\nLegacy tool calls:", legacy_result) # Display the normalized result

empty_message = AIMessage(content="No tool call was made.") # Create a message without tool calls
empty_generation = ChatGeneration(message=empty_message) # Wrap the empty message

empty_result = parser.parse_result([empty_generation]) # Parse the message without tool calls
print("\nNo tool calls:", empty_result) # Display an empty list

empty_first_result = first_parser.parse_result([empty_generation]) # Request the first missing call
print("No first tool call:", empty_first_result) # Display None

try: # Start unsupported-generation error handling
    parser.parse_result([Generation(text="Normal text output")]) # Pass a non-chat generation
except OutputParserException as error: # Catch the expected parser exception
    print("\nGeneration error:", error) # Display the parser error

try: # Start unsupported text-only parsing handling
    parser.parse('{"tool_calls": []}') # Call the unsupported parse method
except NotImplementedError: # Catch the expected unsupported-operation error
    print("parse() is not supported by JsonOutputToolsParser.") # Explain the result

# `JsonOutputKeyToolsParser: JsonOutputToolsParser`

Filters parsed tool calls by a selected tool type.

## Fields

```python
key_name: str # Tool type to retain
```

## Constructor

```python
JsonOutputKeyToolsParser(
    *,
    strict: bool = False, # Value forwarded to JSON decoding strictness
    return_id: bool = False, # Whether returned results retain complete call dictionaries
    first_tool_only: bool = False, # Whether to return only the first matching call
    key_name: str, # Tool type to retain
) -> None
```

## Methods

### `parse_result`

Parses tool calls and retains calls whose `"type"` equals `key_name`.

```python
parse_result(
    self,
    result: list[Generation], # Candidate generations for one model input
    *,
    partial: bool = False, # Whether to accept incomplete JSON arguments
) -> Any # Matching arguments, calls, list, or None
```

The first result must be a `ChatGeneration`; otherwise, the method raises `OutputParserException`.

The method reads modern `AIMessage.tool_calls` when available. Otherwise, it uses the backward-compatible `additional_kwargs["tool_calls"]` path, calls `parse_tool_calls()`, and renames each parsed call's `"name"` key to `"type"`.

When no legacy tool-call data is available, the method returns `None` if `first_tool_only=True`; otherwise, it returns an empty list.

When `first_tool_only=True`, only the first call whose `"type"` equals `key_name` is considered:

- With `return_id=True`, the complete matching dictionary is returned.
- With `return_id=False`, only its `"args"` value is returned.
- When no call matches, `None` is returned.

When `first_tool_only=False`, all matching calls are returned:

- With `return_id=True`, complete matching dictionaries are returned.
- With `return_id=False`, only their `"args"` values are returned.

In [ ]:
from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_tools import JsonOutputKeyToolsParser # Import the real key-based tool parser
from langchain_core.outputs import ChatGeneration # Import ChatGeneration for parse_result


message = AIMessage( # Create an AI message containing multiple tool calls
    content="", # Leave normal message content empty
    tool_calls=[ # Add modern LangChain tool-call dictionaries
        { # Define the first weather tool call
            "name": "get_weather", # Provide the tool name
            "args": {"city": "Delhi", "unit": "celsius"}, # Provide parsed tool arguments
            "id": "call_weather_1", # Provide the tool-call ID
            "type": "tool_call", # Identify this as a tool call
        },
        { # Define a calculator tool call
            "name": "calculate", # Provide the tool name
            "args": {"first": 10, "second": 5, "operation": "multiply"}, # Provide calculator arguments
            "id": "call_calculate_1", # Provide the tool-call ID
            "type": "tool_call", # Identify this as a tool call
        },
        { # Define the second weather tool call
            "name": "get_weather", # Provide the same tool name
            "args": {"city": "Mumbai", "unit": "celsius"}, # Provide another set of arguments
            "id": "call_weather_2", # Provide the second tool-call ID
            "type": "tool_call", # Identify this as a tool call
        },
    ],
) # Finish creating the message

generation = ChatGeneration(message=message) # Wrap the message in a chat generation

arguments_parser = JsonOutputKeyToolsParser( # Create a parser that returns matching arguments
    key_name="get_weather", # Keep only get_weather tool calls
    return_id=False, # Return only each call's args dictionary
    first_tool_only=False, # Return all matching calls
) # Finish creating the parser

weather_arguments = arguments_parser.parse_result([generation]) # Parse all matching tool calls
print("All weather arguments:", weather_arguments) # Display only the matching arguments

invoke_result = arguments_parser.invoke(message) # Parse through the runnable interface
print("Invoke result:", invoke_result) # Display the runnable result

async_result = await arguments_parser.ainvoke(message) # Parse asynchronously in Jupyter
print("Async result:", async_result) # Display the asynchronous result

full_calls_parser = JsonOutputKeyToolsParser( # Create a parser that retains complete calls
    key_name="get_weather", # Keep only get_weather calls
    return_id=True, # Return complete calls including IDs
    first_tool_only=False, # Return all matching calls
) # Finish creating the parser

full_weather_calls = full_calls_parser.parse_result([generation]) # Parse complete matching calls
print("\nWeather calls with IDs:", full_weather_calls) # Display full matching dictionaries

first_parser = JsonOutputKeyToolsParser( # Create a parser that returns one matching call
    key_name="get_weather", # Keep only get_weather calls
    return_id=True, # Retain the selected call's ID
    first_tool_only=True, # Return only the first matching call
) # Finish creating the parser

first_weather_call = first_parser.parse_result([generation]) # Select the first matching call
print("\nFirst weather call:", first_weather_call) # Display the selected call

calculator_parser = JsonOutputKeyToolsParser( # Create a parser for calculator arguments
    key_name="calculate", # Keep only calculate calls
    return_id=False, # Return only the args dictionary
    first_tool_only=True, # Return the first matching call
) # Finish creating the parser

calculator_arguments = calculator_parser.parse_result([generation]) # Parse calculator arguments
print("Calculator arguments:", calculator_arguments) # Display the selected arguments

missing_parser = JsonOutputKeyToolsParser( # Create a parser for a missing tool type
    key_name="send_email", # Request a tool type not present
    return_id=False, # Return only arguments when found
    first_tool_only=True, # Return one match or None
) # Finish creating the parser

missing_result = missing_parser.parse_result([generation]) # Search for the missing tool
print("Missing tool result:", missing_result) # Display None because no call matches

# `PydanticToolsParser: JsonOutputToolsParser`

Parses tool-call arguments into Pydantic model instances.

Both Pydantic v2 and Pydantic v1 model classes are supported.

## Fields

```python
tools: Annotated[list[TypeBaseModel], SkipValidation()] # Pydantic tool models available for parsing
```

## Constructor

```python
PydanticToolsParser(
    *,
    strict: bool = False, # Value forwarded to JSON decoding strictness
    return_id: bool = False, # Whether the parent parser retains tool-call IDs
    first_tool_only: bool = False, # Whether to return only the first parsed model
    tools: Annotated[list[TypeBaseModel], SkipValidation()], # Pydantic tool models
) -> None
```

## Methods

### `parse_result`

Parses tool calls and validates their argument dictionaries with matching Pydantic models.

```python
parse_result(
    self,
    result: list[Generation], # Chat generations containing tool calls
    *,
    partial: bool = False, # Whether to skip incomplete or invalid model data
) -> Any # Pydantic model, list of models, None, or an empty list
```

The method first delegates JSON tool-call parsing to `JsonOutputToolsParser`.

When no JSON results are produced, it returns `None` if `first_tool_only=True`; otherwise, it returns an empty list.

Pydantic v2 model names are determined from `model_config["title"]` when truthy, otherwise from the class name. Pydantic v1 models use their class names. These names are matched against each parsed result's `"type"` value.

Each result's `"args"` value must be a dictionary. With `partial=True`, non-dictionary arguments are skipped. Otherwise, they raise `ValueError`.

An unknown tool type raises `OutputParserException` listing the available model names, or `"<no_tools>"` when none are available.

Matching models are instantiated with `tool(**res["args"])`.

When model construction raises `ValidationError` or `ValueError`:

- With `partial=True`, that result is skipped.
- With `partial=False`, the exception is re-raised.
- Before re-raising, if any supplied chat generation has `response_metadata["stop_reason"] == "max_tokens"`, the parser logs that the output is likely incomplete.

When `first_tool_only=True`, the first successfully constructed model is returned, or `None` when none were constructed. Otherwise, all successfully constructed models are returned as a list.

Partial streaming currently emits a model only after all fields required for successful Pydantic construction are available.

In [ ]:
import json # Import JSON support for encoding tool arguments

from pydantic import BaseModel, ValidationError # Import Pydantic model and validation error

from langchain_core.exceptions import OutputParserException # Import the real LangChain parser exception
from langchain_core.messages import AIMessage # Import a real LangChain AI message
from langchain_core.output_parsers.openai_tools import PydanticToolsParser # Import the real Pydantic tools parser
from langchain_core.outputs import ChatGeneration # Import ChatGeneration for parse_result


class WeatherRequest(BaseModel): # Define the Pydantic model for a weather tool
    city: str # Store the requested city
    unit: str # Store the requested temperature unit


class CalculatorRequest(BaseModel): # Define the Pydantic model for a calculator tool
    first_number: float # Store the first number
    second_number: float # Store the second number
    operation: str # Store the requested mathematical operation


message = AIMessage( # Create an AI message containing raw OpenAI-style tool calls
    content="", # Leave normal message content empty
    additional_kwargs={ # Add provider-specific tool-call data
        "tool_calls": [ # Add multiple raw tool calls
            { # Define the weather tool call
                "id": "call_weather_1", # Provide the tool-call ID
                "type": "function", # Identify the provider tool-call type
                "function": { # Define the called function
                    "name": "WeatherRequest", # Match the WeatherRequest model name
                    "arguments": json.dumps({ # Encode the arguments as JSON
                        "city": "Delhi", # Provide the requested city
                        "unit": "celsius", # Provide the temperature unit
                    }),
                },
            },
            { # Define the calculator tool call
                "id": "call_calculator_1", # Provide the tool-call ID
                "type": "function", # Identify the provider tool-call type
                "function": { # Define the called function
                    "name": "CalculatorRequest", # Match the CalculatorRequest model name
                    "arguments": json.dumps({ # Encode the arguments as JSON
                        "first_number": 10, # Provide the first number
                        "second_number": 5, # Provide the second number
                        "operation": "multiply", # Provide the operation
                    }),
                },
            },
        ],
    },
)

generation = ChatGeneration(message=message) # Wrap the message in a chat generation

parser = PydanticToolsParser( # Create a parser for both tool models
    tools=[WeatherRequest, CalculatorRequest], # Register the available models
    strict=False, # Use normal JSON decoding
    first_tool_only=False, # Return all successfully parsed models
)

models = parser.parse_result([generation]) # Parse and validate both tool calls
print("Parsed models:", models) # Display the Pydantic models

for model in models: # Iterate over the parsed models
    print("Model type:", type(model).__name__) # Display the selected model class
    print("Model data:", model.model_dump()) # Display the validated model data

invoke_result = parser.invoke(message) # Parse through the runnable interface
print("\nInvoke result:", invoke_result) # Display the runnable result

async_result = await parser.ainvoke(message) # Parse asynchronously in Jupyter
print("Async result:", async_result) # Display the asynchronous result

first_parser = PydanticToolsParser( # Create a parser that returns one model
    tools=[WeatherRequest, CalculatorRequest], # Register the available models
    first_tool_only=True, # Return only the first successfully parsed model
)

first_model = first_parser.parse_result([generation]) # Parse the first successful tool call
print("\nFirst model:", first_model) # Display the first model
print("First model type:", type(first_model).__name__) # Display its class

partial_message = AIMessage( # Create a tool call with missing required data
    content="", # Leave normal message content empty
    additional_kwargs={ # Add provider-specific tool-call data
        "tool_calls": [
            {
                "id": "call_partial_1", # Provide the tool-call ID
                "type": "function", # Identify the tool-call type
                "function": {
                    "name": "WeatherRequest", # Select WeatherRequest
                    "arguments": '{"city": "Delhi"}', # Omit the required unit
                },
            },
        ],
    },
)

partial_generation = ChatGeneration(message=partial_message) # Wrap the incomplete message

partial_result = parser.parse_result( # Parse incomplete model data
    [partial_generation], # Provide the incomplete generation
    partial=True, # Skip calls that cannot yet form a valid model
)

print("\nPartial result:", partial_result) # Display an empty list

try: # Start complete validation-error handling
    parser.parse_result([partial_generation]) # Parse incomplete data as complete
except ValidationError as error: # Catch the Pydantic validation error
    print("Complete validation error:", error.errors()[0]["msg"]) # Display the first error

unknown_message = AIMessage( # Create an unregistered tool call
    content="", # Leave normal message content empty
    additional_kwargs={
        "tool_calls": [
            {
                "id": "call_unknown_1", # Provide the tool-call ID
                "type": "function", # Identify the tool-call type
                "function": {
                    "name": "SendEmailRequest", # Use an unregistered model name
                    "arguments": '{"recipient": "user@example.com"}', # Provide valid JSON
                },
            },
        ],
    },
)

unknown_generation = ChatGeneration(message=unknown_message) # Wrap the unknown tool call

try: # Start unknown-tool error handling
    parser.parse_result([unknown_generation]) # Try parsing an unregistered tool
except OutputParserException as error: # Catch the LangChain parser exception
    print("Unknown tool error:", error) # Display the available-tool error